# Initial Raw Data Processing

Download UC Faculty zip and put "UC AWR_website_data with Home Dept 2013 -2024 (1).xlsx" into the colab to start


In [1]:
# Lossless data processing for the UC Faculty Dataset provided by a CPRA request to the UC System

import polars as pl

print("Scanning xlsx files...")


df_dict = pl.read_excel("../data/raw/UC AWR_website_data with Home Dept 2013 -2024 (1).xlsx",sheet_id=0)
df = pl.concat(df_dict.values())
df = df.rename(str.lower)

df = df.with_columns(
    pl.selectors.string().str.to_lowercase()
).with_columns(
    pl.selectors.string().str.strip_chars(),
).with_columns(
    # Since we just have year I figure datetime obj isnt worth it
    pl.col('year').cast(pl.Int16),

    # Change ints
    pl.col('base_pay').cast(pl.Int32),
    pl.col('overtime_pay').cast(pl.Int32),
    pl.col('other_pay').cast(pl.Int32),
    pl.col('gross_pay').cast(pl.Int32),


    # Cast to Categorical
    pl.col('campus').cast(pl.Categorical),
    pl.col('department').cast(pl.Categorical),
)


df.describe()

Scanning xlsx files...


statistic,year,campus,last_name,first_name,title_name,department,gross_pay,base_pay,overtime_pay,other_pay
str,f64,str,str,str,str,str,f64,f64,f64,f64
"""count""",3.660661e6,"""3660661""","""3660661""","""3660660""","""3660657""","""3660320""",3.660661e6,3.660661e6,3.660661e6,3.660661e6
"""null_count""",0.0,"""0""","""0""","""1""","""4""","""341""",0.0,0.0,0.0,0.0
"""mean""",2018.764224,null,null,null,null,null,56705.176119,48504.964208,847.00437,7353.209968
"""std""",3.458144,null,null,null,null,null,78820.283732,55530.470437,3942.074732,39005.584722
"""min""",2013.0,null,"""""","""""","""9964 - no description found""",null,1.0,-385300.0,-49939.0,-345255.0
"""25%""",2016.0,null,null,null,null,null,5166.0,4508.0,0.0,0.0
"""50%""",2019.0,null,null,null,null,null,34225.0,31786.0,0.0,0.0
"""75%""",2022.0,null,null,null,null,null,76860.0,71906.0,0.0,2500.0
"""max""",2024.0,null,"""zyzik""","""ângela""","""zone mech""",null,7.061667e6,1.965054e6,298933.0,6.761667e6


In [2]:
df.head()

year,campus,last_name,first_name,title_name,department,gross_pay,base_pay,overtime_pay,other_pay
i16,cat,str,str,str,cat,i32,i32,i32,i32
2024,"""asucla""","""abbassi""","""pouria""","""dir exec""","""executive director""",390489,356085,0,34405
2024,"""asucla""","""mehdian""","""kamran""","""dir""","""systems development""",247343,236450,0,10893
2024,"""asucla""","""baker""","""donna""","""dir""","""financial planning & analysis""",243072,232372,0,10700
2024,"""asucla""","""moyer""","""michelle""","""dir""","""human resources""",241336,231051,0,10285
2024,"""asucla""","""bolton""","""cynthia""","""dir""","""rest operations""",182084,174324,0,7760


### Load Data

In [3]:
import duckdb
duckdb.connect()


### Role Column
No rows are dropped. `role` comes from `title_name` using the `UC_ROLES` rules in the next cell.
Table name for SQL statements is faculty_salaries

In [4]:
UC_ROLES = [
    ("recalled_emeriti", r"recall|emerit"),
    ("teaching_assistant", r"^teachg (asst|fellow)|^teaching assistant|^(special )?reader|tutor|^assoc in |^lang(uage)? asst"),
    ("grad_student_researcher", r"^gsr"),
    ("postdoc", r"postdoc"),
    ("undergrad_student_worker", r"^stdt"),
    ("medical_resident", r"^resid phys|^intern phys|^chief resid|resid-vet|pharmacy resid"),
    ("ladder_faculty", r"^(act |acting )?(asst |assoc )?prof-(ay|fy|hcomp)(-b/e/e|-law)?(-1/9|-1/10)?$"
                       r"|^(sr )?lect p?soe|prof of teaching|teaching prof|^(asst |assoc )?professor of law$"),
    ("lecturer", r"^(sr )?lect[ -]|^lecturer"),
    ("clinical_adjunct_faculty", r"\bprof\b|prof-|professor|clin instr|adj instr"),
    ("extension_instructor", r"^teacher-unex|^teacher-spec prog"),
    ("researcher_specialist_librarian", r"^(jr |asst |assoc )?specialist|proj(ect)? scientist|^(asst |assoc )?res-"
                                        r"|^(asst |assoc )?researcher|agron|coop(erative)? ext|librarian|^acad coord|^academic coord"),
]


def classify(title: pl.Expr, rules) -> pl.Expr:
    t = title.str.to_lowercase().str.strip_chars()
    expr = pl.lit("staff")
    for role, pattern in reversed(rules):
        expr = pl.when(t.str.contains(pattern)).then(pl.lit(role)).otherwise(expr)
    return expr


df = df.with_columns(classify(pl.col("title_name"), UC_ROLES).alias("role"))

duckdb.sql(f"""
CREATE OR REPLACE TABLE faculty_salaries AS
    SELECT *
    FROM df
    """)

Data info after adding roles

In [5]:
df.describe()

statistic,year,campus,last_name,first_name,title_name,department,gross_pay,base_pay,overtime_pay,other_pay,role
str,f64,str,str,str,str,str,f64,f64,f64,f64,str
"""count""",3.660661e6,"""3660661""","""3660661""","""3660660""","""3660657""","""3660320""",3.660661e6,3.660661e6,3.660661e6,3.660661e6,"""3660661"""
"""null_count""",0.0,"""0""","""0""","""1""","""4""","""341""",0.0,0.0,0.0,0.0,"""0"""
"""mean""",2018.764224,null,null,null,null,null,56705.176119,48504.964208,847.00437,7353.209968,null
"""std""",3.458144,null,null,null,null,null,78820.283732,55530.470437,3942.074732,39005.584722,null
"""min""",2013.0,null,"""""","""""","""9964 - no description found""",null,1.0,-385300.0,-49939.0,-345255.0,"""clinical_adjunct_faculty"""
"""25%""",2016.0,null,null,null,null,null,5166.0,4508.0,0.0,0.0,null
"""50%""",2019.0,null,null,null,null,null,34225.0,31786.0,0.0,0.0,null
"""75%""",2022.0,null,null,null,null,null,76860.0,71906.0,0.0,2500.0,null
"""max""",2024.0,null,"""zyzik""","""ângela""","""zone mech""",null,7.061667e6,1.965054e6,298933.0,6.761667e6,"""undergrad_student_worker"""


In [6]:
df.head()

year,campus,last_name,first_name,title_name,department,gross_pay,base_pay,overtime_pay,other_pay,role
i16,cat,str,str,str,cat,i32,i32,i32,i32,str
2024,"""asucla""","""abbassi""","""pouria""","""dir exec""","""executive director""",390489,356085,0,34405,"""staff"""
2024,"""asucla""","""mehdian""","""kamran""","""dir""","""systems development""",247343,236450,0,10893,"""staff"""
2024,"""asucla""","""baker""","""donna""","""dir""","""financial planning & analysis""",243072,232372,0,10700,"""staff"""
2024,"""asucla""","""moyer""","""michelle""","""dir""","""human resources""",241336,231051,0,10285,"""staff"""
2024,"""asucla""","""bolton""","""cynthia""","""dir""","""rest operations""",182084,174324,0,7760,"""staff"""


## Department Bucketing


### Amount of Existing Departments & Titles


In [7]:
duckdb.sql("""
SELECT COUNT(DISTINCT department) AS unique_departments
FROM faculty_salaries
           """).pl()

unique_departments
i64
10166


## Sample of the roles added for auditing

In [8]:
pl.Config.set_tbl_rows(20)

duckdb.sql("""
SELECT COUNT(department) AS count, department, role
FROM faculty_salaries
GROUP BY department, role
LIMIT 200
           """).pl()

count,department,role
i64,str,str
302,"""som student affairs dept""","""staff"""
2471,"""academic resource center""","""staff"""
63,"""student technology support dep""","""undergrad_student_worker"""
784,"""recreation:student rec center""","""undergrad_student_worker"""
100,"""soe deans office dept""","""staff"""
59,"""dining services""","""teaching_assistant"""
687,"""computer science & engineering""","""undergrad_student_worker"""
259,"""university library dept""","""undergrad_student_worker"""
121,"""unex education""","""extension_instructor"""


### Checking the amount of departments in 

In [9]:

duckdb.sql("""
SELECT COUNT(*) AS count, department,
FROM faculty_salaries
GROUP BY department
ORDER BY count DESC
LIMIT 20
           """).pl()

count,department
i64,str
45646,"""ucdmc medical center"""
28988,"""intercollegiate athletics"""
27965,"""housing/dining/hospitality"""
25286,"""housing & dining services"""
23640,"""mathematics"""
22189,"""medicine"""
22002,"""on campus housing"""
21122,"""medicine-dept administration"""
17923,"""psychology"""


In [10]:
# Analyze Departments
duckdb.sql("""
SELECT DISTINCT department
FROM faculty_salaries
           """).pl()# .write_csv("Departments.csv")

department
str
"""rest operations"""
"""jimmy's"""
"""distribution center"""
"""graduation etc."""
"""bookzone"""
"""commissary"""
"""sales audit"""
"""hss dental dept"""
"""taco bell"""


### Creating The Proposed Buckets

* Engineering
* Computer Science
* Science & Mathematics
* Medicine & Heath Sciences
* Business, Economics & Finance
* Agriculture, Food & Animal
* Architecture & Environmental Design
* Humanities & Languages
* Social Sciences & Communication
* Education & Public Administration

In [11]:
bucket_sql = """
SELECT *,
CASE
    WHEN
        department ILIKE '%computer%'
        OR department ILIKE '%comp sci%'
        OR department ILIKE '%cmptr sci%'
        OR department ILIKE '%computing%'
        OR department ILIKE '%software%'
        OR department ILIKE '%informatics%'
        OR department ILIKE '%cyber%'
        OR department ILIKE '%data science%'
        OR department ILIKE '%information studies%'
        OR department ILIKE '%school of info%'
    THEN 'Computer Science'

    WHEN
        department ILIKE '%medicine%'
        OR department ILIKE '%surgery%'
        OR department ILIKE '%vm:%'
        OR department ILIKE '%med:%'
        OR department ILIKE '%!_%' ESCAPE '!' -- For some reason all of the _'s for UC's are medicine or health lol
        OR department ILIKE '%nurs%'
        OR department ILIKE '%dentistry%'
        OR department ILIKE '%pharma%'
        OR department ILIKE '%gynecology%'
        OR department ILIKE '%urology%'
        OR department ILIKE '%oncology%'
        OR department ILIKE '%family med%'
        OR department ILIKE '%radiation%'
        OR department ILIKE '%ophthalmology%'
        OR department ILIKE '%pathology%'
        OR department ILIKE '%peds%'
        OR department ILIKE '%dermatology%'
        OR department ILIKE '%pediatrics%'
        OR department ILIKE '%anesthesiology%'
        OR department ILIKE '%optometry%'
        OR department ILIKE '%psychiatry%'
        OR department ILIKE '%emerg med%'
        OR department ILIKE '%radiolog%'
        OR department ILIKE '%radiation%'
        OR department ILIKE '%anesthes%'
        OR department ILIKE '%public health%'
        OR department ILIKE '%med ed administration%'
        OR department ILIKE '%deans office-school of medicin%'
        OR department ILIKE '%semel institute%'
        OR department ILIKE '%school of med%'
        OR department ILIKE '%dom cardiology%'
        OR department ILIKE '%ophth clinical%'
        OR department ILIKE '%med-campus%'
        OR department ILIKE '%dntl sci%'
        OR department ILIKE '%ob/gyn%'
        OR department ILIKE '%ambulatory%'
        OR department ILIKE '%electronic health record%'
        OR department ILIKE '%electrocardiology%'
        OR department ILIKE '%epidemiology%'
        OR department ILIKE '%otolaryngology%'
        OR department ILIKE '%hematology%'
        OR department ILIKE '%school of public health%'
        OR department ILIKE '%health policy and management%'
        OR department ILIKE '%health sciences%'
        OR department ILIKE '%lppi: instr & research%'
        OR department ILIKE 'dom %'
        OR department ILIKE 'som %'
        OR department ILIKE 'som-%'
        OR department ILIKE '%obgyn%'
        OR department ILIKE '%sfgh%'
        OR department ILIKE 'psych-%'
        OR department ILIKE 'fcm-%'
    THEN 'Medicine & Health Sciences'

    WHEN
        department ILIKE '%business%'
        OR department ILIKE '%econ%'
        OR department ILIKE '%finance%'
        OR department ILIKE '%of managemen%'
        OR department ILIKE '%haas core%'
        OR department ILIKE '%technology management program%'
        OR department ILIKE '%grad.sch. int.relat./pac.stud.%'
    THEN 'Business, Economics & Finance'

    WHEN
        department ILIKE '%engineer%'
        OR department ILIKE '%engr%'
        OR department ILIKE '%mech%'
        OR department ILIKE '%civil%'
        OR department ILIKE '%areo%'
        OR department ILIKE '%electric%'
        OR department ILIKE '%soe-ams%'
        OR department ILIKE '%chemical%'
        OR department ILIKE '%industrial%'
        OR department ILIKE '%material%'
        OR department ILIKE '%soece-ee-tm-cm%'
    THEN 'Engineering'

    WHEN
        department ILIKE '%math%'
        OR department ILIKE '%stat%'
        OR department ILIKE '%statistics%'
        OR department ILIKE '%bio%'
        OR department ILIKE '%neuro%'
        OR department ILIKE '%phys%'
        OR department ILIKE '%land air & water resources%'
        OR department ILIKE '%climate/atmos sci/phy oceanog%'
        OR department ILIKE '%ocean sciences%'
        OR department ILIKE '%entomology%'
        OR department ILIKE '%physics%'
        OR department ILIKE '%astro%'
        OR department ILIKE '%chem%'
        OR department ILIKE '%sio dep%'
        OR department ILIKE '%molecular, cell%'
        OR department ILIKE '%earth%'
        OR department ILIKE '%oceanic%'
        OR department ILIKE '%ecology%'
        OR department ILIKE '%genetics%'
        OR department ILIKE '%igpp%'
        OR department ILIKE '%geosciences%'
        OR department ILIKE '%oceanography%'
        OR department ILIKE '%geology%'
        OR department ILIKE '%viticulture%'
        OR department ILIKE '%nematology%'
        OR department ILIKE '%demography%'
    THEN 'Science & Mathematics'

    WHEN
        department ILIKE '%plant science%'
        OR department ILIKE '%agriculture%'
        OR department ILIKE '%espm%'
        OR department ILIKE '%nutrition%'
        OR department ILIKE '%inst of environment and sustai%'
        OR department ILIKE '%energy & resources group%'
        OR department ILIKE '%animal%'
        OR department ILIKE '%food science%'
        OR department ILIKE '%food safety%'
    THEN 'Agriculture, Food & Animal'

    WHEN
        department ILIKE '%architec%'
        OR department ILIKE '%planning%'
        OR department ILIKE '%urban%'
        OR department ILIKE '%landscape arch%'
        OR department ILIKE '%environmental%'
    THEN 'Architecture & Environmental Design'

    WHEN
        department ILIKE '%language%'
        OR department ILIKE '%linguistics%'
        OR department ILIKE '%humanities%'
        OR department ILIKE '%english%'
        OR department ILIKE '%literature%'
        OR department ILIKE '%writing%'
        OR department ILIKE '%history%'
        OR department ILIKE '%philosophy%'
        OR department ILIKE '%french%'
        OR department ILIKE '%spanish%'
        OR department ILIKE '%german%'
        OR department ILIKE '%japanese%'
        OR department ILIKE '%dance%'
        OR department ILIKE '%african%'
        OR department ILIKE '%music%'
        OR department ILIKE '%division of arts%'
        OR department ILIKE '%classics%'
        OR department ILIKE '%film%'
        OR department ILIKE '%visual arts%'
        OR department ILIKE '%theatre%'
        OR department ILIKE '%religious%'
        OR department ILIKE '%theater%'
        OR department ILIKE '%asian lang%'
        OR department ILIKE '%eastern lang%'
        OR department ILIKE '%world arts%'
        OR department ILIKE '%southeast asian%'
        OR department ILIKE '%drama department%'
        OR department ILIKE '%department of chicana%'
        OR department ILIKE '%asian-american%'
        OR department ILIKE '%asian%'
        OR department ILIKE '%studio art%'
        OR department ILIKE '%art%'
        OR department ILIKE '%design program%'
        OR department ILIKE '%rhetoric%'
        OR department ILIKE '%near eastern%'
        OR department ILIKE '%chicano%'
        OR department ILIKE '%department of design%'
        OR department ILIKE '%media%'
        OR department ILIKE '%institute for society and gene%'
        OR department ILIKE '%college of creative studies%'
        OR department ILIKE '%american studies%'
        OR department ILIKE '%italian studies%'
        OR department ILIKE '%native american studies%'
        OR department ILIKE '%performance, play & design%'
        OR department ILIKE '%science & technology studies%'
        OR department ILIKE '%drama%'
        OR department ILIKE '%hispanic%'
        OR department ILIKE '%italian%'
        OR department ILIKE '%scandinavian%'
        OR department ILIKE '%slavic%'
        OR department ILIKE '%greek & roman%'
    THEN 'Humanities & Languages'

    WHEN
        department ILIKE '%anthropology%'
        OR department ILIKE '%social science%'
        OR department ILIKE '%sociology%'
        OR department ILIKE '%political science%'
        OR department ILIKE '%politics%'
        OR department ILIKE '%ethnic stud%'
        OR department ILIKE '%women%'
        OR department ILIKE '%psychology%'
        OR department ILIKE '%psychological%'
        OR department ILIKE '%cognitive science%'
        OR department ILIKE '%global & intl studies%'
        OR department ILIKE '%soc sci undergrad%'
        OR department ILIKE '%comm%'
        OR department ILIKE '%gender%'
        OR department ILIKE '%child%'
        OR department ILIKE '%geography%'
        OR department ILIKE '%human ecology%'
        OR department ILIKE '%social welfare%'
        OR department ILIKE '%global & international studies%'
        OR department ILIKE '%african am studies%'
        OR department ILIKE '%black studies%'
        OR department ILIKE '%studies%'
    THEN 'Social Sciences & Communication'

    WHEN
        department ILIKE '%education%'
        OR department ILIKE '%public policy%'
        OR department ILIKE '%policy&strategy%'
        OR department ILIKE 'jd program'
        OR department ILIKE '%law%'
        OR department ILIKE '%legal%'
        OR department ILIKE '%journalism%'
        OR department ILIKE '%teaching%'
    THEN 'Education & Public Administration'

    ELSE 'Other'
END AS dep_bucket
FROM faculty_salaries
"""

duckdb.sql(f"""
CREATE OR REPLACE TABLE faculty_salaries AS
{bucket_sql}
""")

In [12]:
# Send UnBucketable Departments to Other
duckdb.sql("""
UPDATE faculty_salaries
SET dep_bucket = 'Other'
WHERE department IN (
    'continuing & professional ed.',
    'faculty',
    'intellectual prop & indus spon',
    'edu/instruction & research',
    'university - davis',
    'elect eng div operations',
    'mae department accounts',
    'administration',
    'merage/instruction & academic',
    'the preuss school',
    'public health administration',
    'instructional development',
    'geffen academy at ucla',
    'gspp department ops',
    'dean-school of env sci & mgmt',
    'continuing & professional ed. ',
    'continuing and professional ed',
    'structural system facility',
    'chancellor ucsf',
    'oid instructional improvement',
    'tep',
    'sph administration            ',
    'information technology svcs',
    'ofc of instructional developme',
    'geffen academy at ucla        ',
    '',
)
""")

# How many departments are accounted for in each bucket


In [13]:
pl.Config.set_tbl_rows(11)

# How many departments are accounted for in each bucket
duckdb.sql("""
SELECT COUNT(DISTINCT department) AS count, dep_bucket
FROM faculty_salaries
GROUP BY dep_bucket
ORDER BY count DESC
           """).pl()

count,dep_bucket
i64,str
5819,"""Other"""
2329,"""Medicine & Health Sciences"""
478,"""Science & Mathematics"""
457,"""Humanities & Languages"""
340,"""Social Sciences & Communicatio…"
191,"""Engineering"""
177,"""Business, Economics & Finance"""
118,"""Architecture & Environmental D…"
114,"""Education & Public Administrat…"


### Check what department are in the Other bucket

In [14]:
pl.Config.set_tbl_rows(7)
duckdb.sql("""
SELECT COUNT(*) as count, department
FROM faculty_salaries
WHERE dep_bucket = 'Other'
GROUP BY department
HAVING count > 1
ORDER BY count DESC
           """).pl()

count,department
i64,str
45646,"""ucdmc medical center"""
28988,"""intercollegiate athletics"""
27965,"""housing/dining/hospitality"""
25286,"""housing & dining services"""
…,…
2,"""mc3-prenatal diag center-marin"""
2,"""faso student financial aid"""
2,"""uc washing"""


In [15]:
df = duckdb.sql("""
    SELECT *, CONCAT(first_name, ' ', last_name) AS name
    FROM faculty_salaries""").pl()
df = df.drop(["first_name", "last_name"])
df = df.rename({"department": "original_department", "dep_bucket": "department"})
df.write_parquet("../data/processed/golden_uc_2013-2024.parquet")
df.describe()

statistic,year,campus,title_name,original_department,gross_pay,base_pay,overtime_pay,other_pay,role,department,name
str,f64,str,str,str,f64,f64,f64,f64,str,str,str
"""count""",3.660661e6,"""3660661""","""3660657""","""3660320""",3.660661e6,3.660661e6,3.660661e6,3.660661e6,"""3660661""","""3660661""","""3660661"""
"""null_count""",0.0,"""0""","""4""","""341""",0.0,0.0,0.0,0.0,"""0""","""0""","""0"""
"""mean""",2018.764224,null,null,null,56705.176119,48504.964208,847.00437,7353.209968,null,null,null
"""std""",3.458144,null,null,null,78820.283732,55530.470437,3942.074732,39005.584722,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…
"""50%""",2019.0,null,null,null,34225.0,31786.0,0.0,0.0,null,null,null
"""75%""",2022.0,null,null,null,76860.0,71906.0,0.0,2500.0,null,null,null
"""max""",2024.0,"""ucop""","""zone mech""","""young womens program mz""",7.061667e6,1.965054e6,298933.0,6.761667e6,"""undergrad_student_worker""","""Social Sciences & Communicatio…","""ângela gomes tomaz"""
